# Silver 1: parse the PDFs and check each parse

Task `silver_parse` of the job `ingest_financebench`. Reads `silver_parts` (the PDFs cut into parts of a few pages, see `split_pdfs`) as a stream, parses each new **part** with `ai_parse_document` **once**, and checks the result.

| Result of the parse | Goes to |
| --- | --- |
| Complete: it has elements and no page errors | `silver_parsed`, one row per part |
| Some pages failed (`Request rate limit exceeded`, `Internal error`) | `silver_log`, marked **retryable**. It is not stored: a part with missing pages would silently lose text. Only that part is retried, not the whole PDF |
| No elements: the part is over 500 pages (not possible after the split), or the conversion failed | `silver_log`, marked **not retryable** |

The checks are in [checks.py](./checks.py) (plain Python, with tests in `tests/`). The tables are defined in [tables.py](./tables.py), and the parse-and-file function is in [parsing.py](./parsing.py). The retry of the failed parses is the next task, `retry_parses`.

**Using the stream's error handling.** A stream records finished micro-batches in its checkpoint. A failed batch does not move the checkpoint, so the next run restarts at that batch: nothing is lost and nothing is skipped. What a restarted batch must not do is parse the same PDFs again (that is billed), so each batch first drops the parts already in `silver_parsed` or `silver_log`. A restart then costs almost nothing.

**Challenge: `ai_parse_document` answers some pages with an error.** In the first full run 3,180 of 49,965 pages (6%), in 55 PDFs, failed with `Request rate limit exceeded` or `Internal error`, because 10 PDFs were parsed at once. **Solution:** split long PDFs into parts of 50 pages (a test PDF that lost 27 pages whole came back complete in parts), check every parse, never store an incomplete one, parse two files at a time (`max_files_per_batch`), and retry the failures later.

**Changes in bronze.** The stream reads `bronze_pdf_parts` with `skipChangeCommits`, so an update or a delete in that table (for example filling in bytes) does not fail it: only new rows are parsed.

In [ ]:
import sys

sys.path.insert(0, "../common")
from names import Names
from silver_tables import create_silver_tables
from parsing import parse_docs

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("bronze_schema", "finance_bench_bronze")
dbutils.widgets.text("silver_schema", "finance_bench_silver")
dbutils.widgets.text("max_files_per_batch", "2")  # files of parts parsed at once (one PDF is one file). Small, so the parser's rate limit is not hit

n = Names(dbutils.widgets.get("catalog"), dbutils.widgets.get("bronze_schema"), dbutils.widgets.get("silver_schema"))
max_files_per_batch = int(dbutils.widgets.get("max_files_per_batch"))

create_silver_tables(spark, n)

In [ ]:
def parse_new_parts(batch_df, batch_id):
    """One micro-batch: skip the parts that are already recorded (a restarted batch costs nothing), parse the rest."""
    known = (
        spark.table(n.parsed).select("path", "part_no")
        .union(spark.table(n.log).filter("stage = 'parse' AND part_no IS NOT NULL").select("path", "part_no"))
    )
    good, bad = parse_docs(spark, n, batch_df.join(known, ["path", "part_no"], "left_anti"))
    print(f"batch {batch_id}: {good} parts parsed, {bad} logged as bad")


(
    spark.readStream.option("maxFilesPerTrigger", max_files_per_batch).option("skipChangeCommits", "true").table(n.parts)
    .writeStream.foreachBatch(parse_new_parts)
    .option("checkpointLocation", n.parse_checkpoint)
    .trigger(availableNow=True)
    .start().awaitTermination()
)

In [ ]:
print("Parts parsed (silver_parsed):", spark.table(n.parsed).count(), "of", spark.table(n.parts).count())
display(spark.table(n.failures).filter("stage = 'parse'").groupBy("retryable").count())